In [1]:
import jax
import jax.numpy as jnp
from jax.scipy.linalg import expm
import numpyro.distributions as dist
import optax


class SDE_IRT_MAP:
    def __init__(self, R=2, K=7, p=2, q=2):
        self.R = R
        self.K = K
        self.p = p
        self.q = q

    # -------------------------
    # Gamma / SDE drift
    # -------------------------
    def _get_gamma(self, l_diag, l_off, w):
        L = jnp.zeros((self.R, self.R))

        # clip exponentials for stability
        L = L.at[jnp.diag_indices(self.R)].set(jnp.exp(jnp.clip(l_diag, -5, 5)))
        L = L.at[1, 0].set(l_off[0])

        S = L @ L.T

        A = jnp.array([
            [0.0, w[0]],
            [-w[0], 0.0]
        ])

        return S + A

    # -------------------------
    # Iterated Laplace
    # -------------------------
    def _iterated_laplace(self, xi_init, pred_cov, log_lik, max_iter=5):

        def body_fun(i, xi):

            g = jax.grad(log_lik)(xi)
            H = jax.hessian(log_lik)(xi)

            prec_pred = jnp.linalg.solve(pred_cov, jnp.eye(self.R))

            prec_post = prec_pred - H
            prec_post = 0.5 * (prec_post + prec_post.T)
            prec_post = prec_post + 1e-4 * jnp.eye(self.R)

            step = jnp.linalg.solve(prec_post, g)

            # damped Newton step
            xi_new = xi + 0.2 * step

            return xi_new

        xi_final = jax.lax.fori_loop(0, max_iter, body_fun, xi_init)

        return xi_final

    # -------------------------
    # Kalman / Laplace filter step
    # -------------------------
    def _kalman_filter_step(self, carry, t_idx, data, params):

        prev_mean, prev_cov = carry

        dt = data['deltat'][t_idx]

        # Prediction step
        Gamma = self._get_gamma(params['l_diag'], params['l_off'], params['w_skew'])

        Phi = expm(-dt * Gamma)

        mu_t = jnp.dot(params['gamma_latent'], data['Z'][t_idx]) * data['time'][t_idx]

        pred_mean = mu_t + Phi @ (prev_mean - mu_t)

        Q = dt * jnp.eye(self.R)

        pred_cov = Phi @ prev_cov @ Phi.T + Q
        pred_cov = 0.5 * (pred_cov + pred_cov.T) + 1e-5 * jnp.eye(self.R)

        # -------------------------
        # Log likelihood
        # -------------------------
        def log_lik(xi):

            f_map = jnp.array([0, 0, 0, 1, 1, 1, 1])
            xi_mapped = xi[f_map]

            eta = jnp.dot(params['beta'], data['X'][t_idx]) + params['lambda'] * xi_mapped

            # Binary items
            logits_bin = params['theta_bin'] + eta[:3]

            ll_bin = jnp.sum(
                dist.Bernoulli(logits=logits_bin)
                .log_prob(data['Y'][t_idx, :3])
                * (1 - data['missing'][t_idx, :3])
            )

            # Ordered items
            ll_ord = 0.0

            for i in range(4):

                k = i + 3

                cutpoints = jnp.cumsum(
                    jnp.exp(jnp.clip(params['theta_ord_raw'][i], -5, 5))
                ) - 5.0

                ll_ord += (
                    dist.OrderedLogistic(
                        predictor=eta[k],
                        cutpoints=cutpoints
                    )
                    .log_prob(data['Y'][t_idx, k])
                    * (1 - data['missing'][t_idx, k])
                )

            return ll_bin + ll_ord

        # -------------------------
        # Iterated Laplace
        # -------------------------

        xi = self._iterated_laplace(pred_mean, pred_cov, log_lik)

        g = jax.grad(log_lik)(xi)
        H = jax.hessian(log_lik)(xi)

        prec_pred = jnp.linalg.solve(pred_cov, jnp.eye(self.R))

        prec_post = prec_pred - H
        prec_post = 0.5 * (prec_post + prec_post.T)
        prec_post = prec_post + 1e-4 * jnp.eye(self.R)

        updated_cov = jnp.linalg.solve(prec_post, jnp.eye(self.R))
        updated_cov = 0.5 * (updated_cov + updated_cov.T)

        updated_mean = xi

        # Laplace likelihood correction
        logdet_pred = jnp.linalg.slogdet(prec_pred)[1]
        logdet_post = jnp.linalg.slogdet(prec_post)[1]

        laplace_term = -0.5 * (logdet_post - logdet_pred)

        ll_step = log_lik(updated_mean) + laplace_term

        return (updated_mean, updated_cov), (updated_mean, updated_cov, ll_step)

    # -------------------------
    # Loss
    # -------------------------
    def loss_fn(self, params, data):

        params = params.copy()

        params['lambda'] = 1 + jax.nn.softplus(params['raw_lambda'])

        init_state = (jnp.zeros(self.R), jnp.eye(self.R))

        _, (_, _, log_liks) = jax.lax.scan(
            lambda c, i: self._kalman_filter_step(c, i, data, params),
            init_state,
            jnp.arange(len(data['time']))
        )

        total_ll = jnp.sum(log_liks)

        # -------------------------
        # Priors
        # -------------------------

        Gamma = self._get_gamma(params['l_diag'], params['l_off'], params['w_skew'])

        l_prior = 0.0

        sigma_theta = jnp.exp(params['log_sigma_theta'])

        l_prior += dist.Normal(0, 5).log_prob(params['mu_theta'])
        l_prior += dist.HalfCauchy(2).log_prob(sigma_theta)

        l_prior += jnp.sum(
            dist.Normal(params['mu_theta'], sigma_theta)
            .log_prob(params['theta_bin'])
        )

        l_prior += jnp.sum(
            dist.Normal(params['mu_theta'], sigma_theta)
            .log_prob(params['theta_ord_raw'])
        )

        l_prior += jnp.sum(dist.Cauchy(0, 2).log_prob(params['beta']))

        l_prior += jnp.sum(dist.Normal(0, 0.5).log_prob(Gamma))

        loss = -(total_ll + l_prior)
        loss = jnp.sum(loss) 

        # NaN guard
        loss = jnp.nan_to_num(loss, nan=1e6, posinf=1e6, neginf=-1e6)

        return loss

    # -------------------------
    # Fit
    # -------------------------
    def fit(self, data, n_iter=4000):

        params = {

            'l_diag': jnp.zeros(self.R),

            'l_off': jnp.zeros(1),

            'w_skew': jnp.zeros(1),

            'beta': jnp.zeros((self.K, self.p)),

            'raw_lambda': jnp.zeros(self.K),

            'mu_theta': jnp.zeros(1),

            'log_sigma_theta': jnp.zeros(1),

            'theta_bin': jnp.zeros(3),

            'theta_ord_raw': jnp.tile(
                jnp.array([jnp.log(1.0), jnp.log(1.0), jnp.log(1.0)]),
                (4, 1)
            ),

            'gamma_latent': jnp.zeros((self.R, self.q)),
        }

        optimizer = optax.adam(learning_rate=3e-4)

        opt_state = optimizer.init(params)

        losses = []

        @jax.jit
        def update_step(p, s):

            loss_val, grads = jax.value_and_grad(self.loss_fn)(p, data)

            updates, s = optimizer.update(grads, s)

            p = optax.apply_updates(p, updates)

            return p, s, loss_val

        for i in range(n_iter):

            params, opt_state, loss = update_step(params, opt_state)

            losses.append(loss)

            if i % 200 == 0:
                print(f"Step {i}, Loss: {loss:.4f}")

        params['lambda'] = 1 + jax.nn.softplus(params['raw_lambda'])

        return params, losses

/u/zwu1/.conda/envs/ou/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
from scipy.linalg import expm as s_expm

def simulate_study(shifted_mean=False, Nsub=600, p=2, q=2):
    K, R = 7, 2
    repme = np.random.randint(2, 13, size=Nsub)
    N = np.sum(repme)
    
    ID = np.repeat(np.arange(1, Nsub + 1), repme)
    cumu = np.cumsum(repme)
    
    X = np.random.normal(size=(N, p))
    Z = np.random.binomial(1, 0.5, size=(N, q)) 
    
    # --- 1. Latent Process Parameters ---
    Gamma = np.array([[0.18, -0.07], 
                      [0.10, 0.15]])
    rho = 0.60
    Omega = np.array([[1.0, rho], [rho, 1.0]])
    
    gamma_latent = np.zeros((R, q))
    if shifted_mean:
        gamma_latent = np.array([[0.8, 1.2], [0.5, 0.9]])

    xi = np.zeros((N, R))
    deltat = np.zeros(N)
    time = np.zeros(N)
    
    for i in range(Nsub):
        start = cumu[i] - repme[i]
        # Ensuring time steps are not too large for stability
        t_subj = np.concatenate(([0], np.cumsum(np.random.uniform(0.5, 1.5, size=repme[i]-1))))
        time[start : start + repme[i]] = t_subj
        
        mu_start = (gamma_latent @ Z[start]) * time[start]
        xi[start] = np.random.multivariate_normal(mu_start, Omega)
        
        for j in range(1, repme[i]):
            k = start + j
            dt = t_subj[j] - t_subj[j-1]
            deltat[k] = dt
            Phi = s_expm(-dt * Gamma)
            
            # --- Robust Q Calculation ---
            Q = Omega - Phi @ Omega @ Phi.T
            # 1. Enforce Symmetry
            Q = 0.5 * (Q + Q.T)
            # 2. Add small ridge for numerical stability
            Q += np.eye(R) * 1e-6 
            
            # 3. Eigenvalue check: if still not PSD, use a small identity floor
            evals = np.linalg.eigvals(Q)
            if np.any(evals <= 0):
                Q += np.eye(R) * (abs(np.min(evals)) + 1e-6)
            
            target_k = (gamma_latent @ Z[k]) * time[k]
            target_prev = (gamma_latent @ Z[k-1]) * time[k-1]
            cond_mean = target_k + Phi @ (xi[k-1] - target_prev)
            
            xi[k] = np.random.multivariate_normal(cond_mean, Q)

    # --- 2. Measurement Model Parameters (Unchanged) ---
    lam = np.array([1.20, 4.00, 4.10, 3.10, 5.20, 3.00, 1.70])
    B = np.zeros((K, p))
    B[1, :] = [0.10, 0.20]   
    B[4, :] = [0.30, -0.30]  

    sig_b = np.zeros(K)
    sig_b[0], sig_b[2], sig_b[3], sig_b[6] = 3.70, 4.80, 3.10, 1.70
    b = np.random.normal(0, 1, size=(Nsub, K)) * sig_b
    
    Y = np.zeros((N, K), dtype=int)
    def inv_logit(x): return 1 / (1 + np.exp(-x))

    for i in range(N):
        sub_idx = ID[i] - 1
        for k in range(K):
            f_idx = 0 if k < 3 else 1
            eta = X[i] @ B[k] + lam[k] * xi[i, f_idx] + b[sub_idx, k]

            if k < 3:
                theta_base = 2.30 if k == 0 else (2.90 if k == 2 else 0.0)
                prob = inv_logit(theta_base + eta)
                Y[i, k] = np.random.binomial(1, prob)
            else:
                if k == 4: th = [-7.50, -2.50, 2.60]
                elif k == 6: th = [-4.30, -1.00, 1.40]
                else: th = [0.0]

                p_cum = [inv_logit(t - eta) for t in th]
                probs = np.diff([0] + p_cum + [1])
                probs = np.clip(probs, 1e-8, 1.0)
                probs /= np.sum(probs)
                Y[i, k] = np.random.choice(np.arange(1, len(probs) + 1), p=probs)

    return {
        'N': N, 'Nsub': Nsub, 'K': K, 'R': R, 'p': p, 'q': q,
        'ID': ID, 'cumu': cumu, 'repme': repme, 'Y': Y,
        'missing_ID': np.zeros((N, K), dtype=int),
        'deltat': deltat, 'time': time, 'X': X, 'Z': Z,
        'ncate4': 2, 'ncate5': 4, 'ncate6': 2, 'ncate7': 4, 
        'true_xi': xi, 'true_gamma': gamma_latent, 
        'true_lambda': lam, 'true_Gamma_mat': Gamma
    }


In [3]:
import pandas as pd
import numpy as np

def export_results_to_csv(fitted_params, true_params, filename="model_results.csv"):
    rows = []
    
    # Flatten the dictionaries to compare parameter by parameter
    for key in true_params.keys():
        true_val = np.array(true_params[key]).flatten()
        est_val = np.array(fitted_params[key]).flatten()
        
        for i in range(len(true_val)):
            t = true_val[i]
            e = est_val[i]
            
            # Calculate metrics
            rbias = (e - t) / t if t != 0 else (e - t)
            mse = (e - t)**2
            
            # Create label (e.g., lambda_0, lambda_1...)
            label = f"{key}_{i}" if len(true_val) > 1 else key
            
            rows.append({
                "Parameter": label,
                "True_Value": round(float(t), 3),
                "Estimated_Value": round(float(e), 3),
                "RB": round(float(rbias), 3),
                "MSE": round(float(mse), 6)
            })
            
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"Results successfully saved to {filename}")
    return df

# Usage (place this after your model.fit() call):


In [4]:
def get_standard_errors(model, fitted_params, data):
    # 1. Flatten the parameters into a single vector
    flat_params, unflatten_fn = jax.flatten_util.ravel_pytree(fitted_params)
    
    def loss_wrapper(flat_p):
        p = unflatten_fn(flat_p)
        return model.loss_fn(p, data)
    
    # 2. Compute the Hessian (Matrix of second derivatives)
    # The inverse of the Hessian at the MAP estimate is the Covariance Matrix
    hessian_map = jax.hessian(loss_wrapper)(flat_params)
    
    # 3. Invert the Hessian to get the Variance-Covariance Matrix
    # We add a small ridge (1e-4) to ensure the matrix is invertible
    cov_matrix = jnp.linalg.inv(hessian_map + jnp.eye(len(flat_params)) * 1e-4)
    
    # 4. Standard Errors are the square root of the diagonal
    se_flat = jnp.sqrt(jnp.maximum(jnp.diag(cov_matrix), 0.0))
    return unflatten_fn(se_flat)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def compile_results(fitted_params, raw_data, model):
    """
    Maps fitted params to true params and calculates RB and MSE.
    """
    rows = []
    
    # --- 1. Map Lambda (Loadings) ---
    true_lam = raw_data['true_lambda']
    est_lam = np.array(fitted_params['lambda'])
    for i in range(len(true_lam)):
        rows.append(calc_metrics(f"lambda_{i+1}", true_lam[i], est_lam[i]))

    # --- 2. Map Gamma (Drift Matrix) ---
    true_Gamma = raw_data['true_Gamma_mat']
    est_Gamma = np.array(model._get_gamma(fitted_params['l_diag'], fitted_params['l_off'], fitted_params['w_skew']))
    # Flattening 2x2 matrix to gamma_11, 12, 21, 22
    for r in range(2):
        for c in range(2):
            rows.append(calc_metrics(f"gamma_{r+1}{c+1}", true_Gamma[r, c], est_Gamma[r, c]))

    # --- 3. Map Beta (Covariates) ---
    # Simulation uses B[1,:] and B[4,:] specifically
    # True B is (K, p) -> (7, 2)
    # We'll check the specific non-zero indices from your simulation
    est_beta = np.array(fitted_params['beta'])
    indices_to_check = [(1,0), (1,1), (4,0), (4,1)] # beta_21, 22, 51, 52
    for r, c in indices_to_check:
        # Note: B in simulation is 0-indexed [1] which is Item 2
        # But Table labels usually use 1-indexing.
        rows.append(calc_metrics(f"beta_{r+1}{c+1}", 0.0, est_beta[r, c])) # True values are 0.1, 0.2 etc.
        # Note: In your sim, B[1,:] = [0.1, 0.2]. Let's manually grab them for the table:
        if r == 1: true_b = 0.1 if c == 0 else 0.2
        if r == 4: true_b = 0.3 if c == 0 else -0.3
        rows[-1]["True"] = true_b

    # --- 4. Map Thresholds (Theta) ---
    # Binary Intercepts (Items 1-3)
    est_theta_bin = np.array(fitted_params['theta_bin'])
    true_theta_bin = [2.30, 0.0, 2.90] # Mapping from sim logic
    for i in range(3):
        rows.append(calc_metrics(f"theta_{i+1}", true_theta_bin[i], est_theta_bin[i]))

    # Ordinal Thresholds (Items 5 and 7)
    # Simulation: k=4 (Item 5) has 3 th; k=6 (Item 7) has 3 th.
    est_theta_ord = np.array(fitted_params['theta_ord_raw'])
    true_th_5 = [-7.50, -2.50, 2.60]
    true_th_7 = [-4.30, -1.00, 1.40]
    
    # Note: fitted_params['theta_ord_raw'] stores log-increments. 
    # We must transform them back to the same scale as 'th' in simulation.
    for i, true_vals in zip([1, 3], [true_th_5, true_th_7]): # indices 1 and 3 in the (4,3) array
        est_th_transformed = np.cumsum(np.exp(est_theta_ord[i])) - 5.0
        item_num = 5 if i == 1 else 7
        for j in range(3):
            rows.append(calc_metrics(f"theta_{item_num}{j+1}", true_vals[j], est_th_transformed[j]))

    df = pd.DataFrame(rows)
    # Re-calculate RB and MSE safely
    df['RB'] = (df['Estimated'] - df['True']) / np.where(df['True'] == 0, 1, np.abs(df['True']))
    df['MSE'] = (df['Estimated'] - df['True'])**2
    return df

def calc_metrics(name, true_val, est_val):
    return {"Para.": name, "True": float(true_val), "Estimated": float(est_val)}

def run_test():
    print("--- Simulating Data ---")
    raw_data = simulate_study(shifted_mean=True, Nsub=600, p=2, q=2)
    
    # ... [Keep your JAX preprocessing logic here] ...

    data_for_model = {
        'Y': jnp.array(raw_data['Y']).at[:, 3:].add(-1), # 0-based
        'X': jnp.array(raw_data['X']),
        'Z': jnp.array(raw_data['Z']),
        'deltat': jnp.array(raw_data['deltat']),
        'time': jnp.array(raw_data['time']),
        'missing': jnp.array(raw_data['missing_ID']),
        'ID': jnp.array(raw_data['ID'] - 1, dtype=jnp.int32)
    }

    print("--- Initializing and Fitting Model ---")
    model = SDE_IRT_MAP(R=2, K=7, p=2, q=2)
    fitted_params, loss_hist = model.fit(data_for_model, n_iter=5000)

    # --- Generate CSV ---
    results_df = compile_results(fitted_params, raw_data, model)
    results_df.to_csv("sde_irt_results.csv", index=False)

    print("\n--- Summary Table ---")
    print(results_df.to_string(index=False))

    plt.figure(figsize=(8, 4))
    plt.plot(loss_hist, color='blue')
    plt.title("Training Loss (Negative Log-Posterior)")
    plt.yscale('log')
    plt.grid(True, which="both", ls="-", alpha=0.5)
    plt.show()

    return raw_data, fitted_params

if __name__ == "__main__":
    raw, params = run_test()

--- Simulating Data ---
--- Initializing and Fitting Model ---
Step 0, Loss: 32734.2168
Step 200, Loss: 30300.8438
Step 400, Loss: 28473.7266
Step 600, Loss: 27047.2715
Step 800, Loss: 25885.3320
Step 1000, Loss: 24917.0293
Step 1200, Loss: 24095.2754
Step 1400, Loss: 23383.2930
Step 1600, Loss: 22752.3652
Step 1800, Loss: 22180.7871
Step 2000, Loss: 21652.3320
Step 2200, Loss: 21154.8496
Step 2400, Loss: 20679.4180
Step 2600, Loss: 20219.8125
Step 2800, Loss: 19772.0273
Step 3000, Loss: 19333.7773
Step 3200, Loss: 18904.0312
Step 3400, Loss: 18482.6914
Step 3600, Loss: 18070.2773
Step 3800, Loss: 17667.6914
Step 4000, Loss: 17276.0547
Step 4200, Loss: 16896.5508
